In [1]:
import pandas as pd

df = pd.read_excel(
    "/home/feliciano/Downloads/00118_Acoustique_Poisson_2026-07-27.xlsx",
    engine="openpyxl"
)

print(
    df["Behavior / Activity"]
    .value_counts(dropna=False)
)

Behavior / Activity
Normal activity             282
Clustering / aggregation    164
Agitation                   156
False                        69
Lethargy                     28
Jumping / splashing          11
NaN                          10
Reactivity to noise           5
Name: count, dtype: int64


In [2]:
import pandas as pd

df = pd.read_excel(
    "/home/feliciano/Downloads/00118_Acoustique_Poisson_2026-07-27.xlsx",
    engine="openpyxl"
)

df = df.dropna(
    subset=["Behavior / Activity"]
)

df = df[
    df["Behavior / Activity"] != False
]

df["start"] = pd.to_datetime(
    df["Date"].astype(str) + " " +
    df["Time entered"].astype(str),
    errors="coerce"
)

df["end"] = pd.to_datetime(
    df["Date"].astype(str) + " " +
    df["Exit time"].astype(str),
    errors="coerce"
)

df["seconds"] = (
    df["end"] - df["start"]
).dt.total_seconds()

print(
    df.groupby(
        "Behavior / Activity"
    )["seconds"].sum() / 2
)

Behavior / Activity
Agitation                   42900.0
Clustering / aggregation    43200.0
Jumping / splashing          2910.0
Lethargy                     4560.0
Normal activity             72990.0
Reactivity to noise          1320.0
Name: seconds, dtype: float64


In [3]:
import os
import shutil
import pandas as pd

ETHOGRAM = "/home/feliciano/Downloads/00118_Acoustique_Poisson_2026-07-27.xlsx"

SEGMENTS_DIR = "/media/feliciano/Aux/AI_AFS_DATASET/segmented_2s"

OUTPUT_DIR = "/media/feliciano/Aux/AI_AFS_DATASET/behavior_dataset"

df = pd.read_excel(
    ETHOGRAM,
    engine="openpyxl"
)

df = df.dropna(
    subset=["Behavior / Activity"]
)

# remove spreadsheet artifacts
df = df[
    df["Behavior / Activity"] != False
]

# merge rare classes
def map_class(x):

    x = str(x)

    if x == "Normal activity":
        return "normal"

    if x == "Agitation":
        return "agitation"

    if x == "Clustering / aggregation":
        return "clustering"

    return "other"

df["class"] = df[
    "Behavior / Activity"
].apply(map_class)

df["start"] = pd.to_datetime(
    df["Date"].astype(str)
    + " "
    + df["Time entered"].astype(str),
    errors="coerce"
)

df["end"] = pd.to_datetime(
    df["Date"].astype(str)
    + " "
    + df["Exit time"].astype(str),
    errors="coerce"
)

intervals = df[
    ["start", "end", "class"]
].dropna()

# create folders
for c in intervals["class"].unique():

    os.makedirs(
        os.path.join(
            OUTPUT_DIR,
            c
        ),
        exist_ok=True
    )

# classify wav files
for wav in os.listdir(SEGMENTS_DIR):

    if not wav.endswith(".wav"):
        continue

    try:

        ts = pd.to_datetime(
            wav.replace(".wav", ""),
            format="%Y-%m-%d_%H-%M-%S"
        )

    except:
        continue

    match = intervals[
        (intervals["start"] <= ts)
        &
        (ts < intervals["end"])
    ]

    if len(match) == 0:
        continue

    label = match.iloc[0]["class"]

    shutil.copy2(
        os.path.join(
            SEGMENTS_DIR,
            wav
        ),
        os.path.join(
            OUTPUT_DIR,
            label,
            wav
        )
    )

print("Done")

Done


In [6]:
import os
import joblib
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

# =====================================================
# DATASET
# =====================================================

DATASET = "/media/feliciano/Aux/AI_AFS_DATASET/behavior_dataset"

CLASSES = [
    "normal",
    "clustering",
    "agitation",
    "other"
]

SR = 16000

# =====================================================
# FEATURE EXTRACTION
# =====================================================

X = []
y = []

for label in CLASSES:

    folder = os.path.join(
        DATASET,
        label
    )

    files = [
        f for f in os.listdir(folder)
        if f.endswith(".wav")
    ]

    print(label, len(files))

    for file in files:

        path = os.path.join(
            folder,
            file
        )

        try:

            signal, sr = librosa.load(
                path,
                sr=SR,
                mono=True
            )

            signal = (
                signal - np.mean(signal)
            ) / (
                np.std(signal) + 1e-8
            )

            features = []

            # RMS
            features.append(
                np.mean(
                    librosa.feature.rms(
                        y=signal
                    )
                )
            )

            # ZCR
            features.append(
                np.mean(
                    librosa.feature.zero_crossing_rate(
                        signal
                    )
                )
            )

            # Spectral Centroid
            features.append(
                np.mean(
                    librosa.feature.spectral_centroid(
                        y=signal,
                        sr=sr
                    )
                )
            )

            # Spectral Bandwidth
            features.append(
                np.mean(
                    librosa.feature.spectral_bandwidth(
                        y=signal,
                        sr=sr
                    )
                )
            )

            # Spectral Rolloff
            features.append(
                np.mean(
                    librosa.feature.spectral_rolloff(
                        y=signal,
                        sr=sr
                    )
                )
            )

            # Spectral Contrast
            features.append(
                np.mean(
                    librosa.feature.spectral_contrast(
                        y=signal,
                        sr=sr
                    )
                )
            )

            # MFCCs
            mfcc = librosa.feature.mfcc(
                y=signal,
                sr=sr,
                n_mfcc=20
            )

            for i in range(20):

                features.append(
                    np.mean(
                        mfcc[i]
                    )
                )

            X.append(features)
            y.append(label)

        except Exception as e:

            print(
                "ERROR:",
                path,
                e
            )

# =====================================================
# NUMPY
# =====================================================

X = np.array(
    X,
    dtype=np.float32
)

y = np.array(y)

print("\nSamples:", len(X))
print("Features:", X.shape[1])

# =====================================================
# LABEL ENCODING
# =====================================================

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print("\nClasses:")

for idx, cls in enumerate(
    encoder.classes_
):
    print(idx, cls)

# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

# =====================================================
# XGBOOST
# =====================================================

model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=len(CLASSES),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

print("\nTraining XGBoost...\n")

model.fit(
    X_train,
    y_train
)

# =====================================================
# PREDICT
# =====================================================

pred = model.predict(
    X_test
)

# =====================================================
# RESULTS
# =====================================================

print(
    "\nAccuracy:",
    accuracy_score(
        y_test,
        pred
    )
)

print(
    "\nClassification Report:\n"
)

print(
    classification_report(
        y_test,
        pred,
        target_names=encoder.classes_
    )
)

print(
    "\nConfusion Matrix:\n"
)

print(
    confusion_matrix(
        y_test,
        pred
    )
)

# =====================================================
# FEATURE IMPORTANCE
# =====================================================

feature_names = [
    "rms",
    "zcr",
    "centroid",
    "bandwidth",
    "rolloff",
    "contrast"
]

for i in range(20):

    feature_names.append(
        f"mfcc_{i+1}"
    )

importance = pd.DataFrame({

    "feature": feature_names,

    "importance":
        model.feature_importances_
})

print(
    "\nTop Features:\n"
)

print(
    importance
    .sort_values(
        "importance",
        ascending=False
    )
    .head(15)
)

# =====================================================
# SAVE
# =====================================================

joblib.dump(
    model,
    "behavior_xgboost.pkl"
)

joblib.dump(
    encoder,
    "behavior_encoder.pkl"
)

print(
    "\nSaved:"
)

print(
    "behavior_xgboost.pkl"
)

print(
    "behavior_encoder.pkl"
)

normal 8049
clustering 1260
agitation 1140
other 240

Samples: 10689
Features: 26

Classes:
0 agitation
1 clustering
2 normal
3 other

Training XGBoost...


Accuracy: 0.9256314312441534

Classification Report:

              precision    recall  f1-score   support

   agitation       0.90      0.57      0.70       228
  clustering       0.93      0.91      0.92       252
      normal       0.93      0.99      0.96      1610
       other       0.81      0.62      0.71        48

    accuracy                           0.93      2138
   macro avg       0.89      0.77      0.82      2138
weighted avg       0.92      0.93      0.92      2138


Confusion Matrix:

[[ 130    8   90    0]
 [   2  230   17    3]
 [  12    5 1589    4]
 [   0    5   13   30]]

Top Features:

      feature  importance
9      mfcc_4    0.134555
14     mfcc_9    0.060912
5    contrast    0.058027
19    mfcc_14    0.052745
10     mfcc_5    0.043913
1         zcr    0.042681
13     mfcc_8    0.039644
25    mfcc_20    

In [9]:
import os
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB

# =====================================================
# DATASET
# =====================================================

DATASET = "/media/feliciano/Aux/AI_AFS_DATASET/behavior_dataset"

CLASSES = [
    "normal",
    "clustering",
    "agitation",
    "other"
]

SR = 16000

# =====================================================
# FEATURE EXTRACTION
# =====================================================

X = []
y = []

for label in CLASSES:

    folder = os.path.join(DATASET, label)

    files = [
        f for f in os.listdir(folder)
        if f.endswith(".wav")
    ]

    print(label, len(files))

    for file in files:

        path = os.path.join(folder, file)

        try:

            signal, sr = librosa.load(
                path,
                sr=SR,
                mono=True
            )

            signal = (
                signal - np.mean(signal)
            ) / (
                np.std(signal) + 1e-8
            )

            features = []

            # RMS
            features.append(
                np.mean(
                    librosa.feature.rms(
                        y=signal
                    )
                )
            )

            # ZCR
            features.append(
                np.mean(
                    librosa.feature.zero_crossing_rate(
                        signal
                    )
                )
            )

            # Spectral Centroid
            features.append(
                np.mean(
                    librosa.feature.spectral_centroid(
                        y=signal,
                        sr=sr
                    )
                )
            )

            # Spectral Bandwidth
            features.append(
                np.mean(
                    librosa.feature.spectral_bandwidth(
                        y=signal,
                        sr=sr
                    )
                )
            )

            # Spectral Rolloff
            features.append(
                np.mean(
                    librosa.feature.spectral_rolloff(
                        y=signal,
                        sr=sr
                    )
                )
            )

            # Spectral Contrast
            features.append(
                np.mean(
                    librosa.feature.spectral_contrast(
                        y=signal,
                        sr=sr
                    )
                )
            )

            # MFCC 1-20
            mfcc = librosa.feature.mfcc(
                y=signal,
                sr=sr,
                n_mfcc=20
            )

            for i in range(20):

                features.append(
                    np.mean(
                        mfcc[i]
                    )
                )

            X.append(features)
            y.append(label)

        except Exception as e:

            print(
                "ERROR:",
                path,
                e
            )

# =====================================================
# NUMPY
# =====================================================

X = np.array(
    X,
    dtype=np.float32
)

y = np.array(y)

print("\nSamples:", len(X))
print("Features:", X.shape[1])

# =====================================================
# LABEL ENCODING
# =====================================================

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print("\nClasses")

for i, cls in enumerate(
    encoder.classes_
):
    print(i, cls)

# =====================================================
# SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

# =====================================================
# MODELS
# =====================================================

models = {

    "KNN": Pipeline([

        ("scaler", StandardScaler()),

        ("model", KNeighborsClassifier(
            n_neighbors=7
        ))

    ]),

    "SVM": Pipeline([

        ("scaler", StandardScaler()),

        ("model", SVC(
            kernel="rbf",
            C=10,
            gamma="scale"
        ))

    ]),

    "Random Forest": RandomForestClassifier(

        n_estimators=300,

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    ),

    "MLP": Pipeline([

        ("scaler", StandardScaler()),

        ("model", MLPClassifier(

            hidden_layer_sizes=(128,64),

            learning_rate_init=0.001,

            max_iter=300,

            random_state=42

        ))

    ]),

    "Naive Bayes": GaussianNB()
}

# =====================================================
# TRAIN/EVALUATE
# =====================================================

results = []

for name, model in models.items():

    print("\n")
    print("=" * 70)
    print(name)
    print("=" * 70)

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(
        X_test
    )

    acc = accuracy_score(
        y_test,
        pred
    )

    results.append({

        "Model": name,

        "Accuracy": acc

    })

    print(
        "\nAccuracy:",
        round(acc, 4)
    )

    print(
        "\nClassification Report:\n"
    )

    print(
        classification_report(
            y_test,
            pred,
            target_names=encoder.classes_
        )
    )

    print(
        "\nConfusion Matrix:\n"
    )

    print(
        confusion_matrix(
            y_test,
            pred
        )
    )

# =====================================================
# FINAL COMPARISON
# =====================================================

results_df = pd.DataFrame(
    results
).sort_values(
    "Accuracy",
    ascending=False
)

print("\n")
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(results_df)

results_df.to_csv(
    "model_comparison.csv",
    index=False
)

print("\nSaved: model_comparison.csv")

normal 8049
clustering 1260
agitation 1140
other 240

Samples: 10689
Features: 26

Classes
0 agitation
1 clustering
2 normal
3 other


KNN

Accuracy: 0.9153

Classification Report:

              precision    recall  f1-score   support

   agitation       0.78      0.53      0.63       228
  clustering       0.93      0.93      0.93       252
      normal       0.93      0.97      0.95      1610
       other       0.85      0.71      0.77        48

    accuracy                           0.92      2138
   macro avg       0.87      0.79      0.82      2138
weighted avg       0.91      0.92      0.91      2138


Confusion Matrix:

[[ 120    7  101    0]
 [   2  235   13    2]
 [  31    7 1568    4]
 [   1    5    8   34]]


SVM

Accuracy: 0.9275

Classification Report:

              precision    recall  f1-score   support

   agitation       0.86      0.59      0.70       228
  clustering       0.91      0.94      0.93       252
      normal       0.94      0.98      0.96      1610
    